In [2]:
import transformers

/home/athankar/miniconda3/envs/redteam/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

In [4]:
a  = torch.zeros(2,3)

In [5]:
a.shape

torch.Size([2, 3])

In [6]:
a.unsqueeze(-1).shape

torch.Size([2, 3, 1])

In [7]:
a.unsqueeze(0).shape

torch.Size([1, 2, 3])

In [8]:
# config = transformers.AutoConfig.from_pretrained(
#         "mistralai/Mistral-7B-Instruct-v0.1",
#         trust_remote_code=True,
#         cache_dir="/data/tir/projects/tir6/bisk/athankar/projects/.cache",
#     )

# model = transformers.AutoModelForCausalLM.from_pretrained(
#         "mistralai/Mistral-7B-Instruct-v0.1",
#         trust_remote_code=True,
#         cache_dir="/data/tir/projects/tir6/bisk/athankar/projects/.cache",
#     )

Loading checkpoint shards: 100%|███████████| 2/2 [00:03<00:00,  2.00s/it]


In [9]:
# out = model(**{
#             "input_ids": torch.ones(size = (2, 4096)).long(),
#             "labels": torch.ones(size = (2, 4096)).long(),
#             "attention_mask": torch.ones(size = (2, 4096)).long(),
#         })

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [11]:
model.__call__

<bound method Module._wrapped_call_impl of MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm

In [10]:
out.keys()

odict_keys(['loss', 'logits', 'past_key_values'])

In [12]:
out['loss']

tensor(21.0105, grad_fn=<NllLossBackward0>)

In [13]:
out["logits"]

tensor([[[-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         ...,
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392]],

        [[-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         ...,
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392],
         [-4.6542, -4.4159, -0.0891,  ..., -3.4539, -2.4985, -3.0392]]],
       grad_fn=<UnsafeViewBackward0>)

In [42]:
logits = out["logits"]
logits = logits[..., :-1, :].contiguous()
labels = torch.ones(size = (2, 4096)).long()
labels = labels[..., 1:].contiguous()


loss_fn = CrossEntropyLoss()
loss_fn(logits.view(-1, 32000), labels.view(-1))

tensor(21.0105, grad_fn=<NllLossBackward0>)

In [57]:


log_probs = -F.log_softmax(logits, dim=-1) # Log softmax over the vocabulary dimension
if labels.dim() == log_probs.dim() - 1:
    labels = labels.unsqueeze(-1) # Adds a dimension to make it broadcastable for the vocabulary dimension?
    # rewards = rewards.unsqueeze(-1) # Adds a dimension to make it broadcastable for the vocabulary dimension?
rewards = torch.ones_like(labels)
padding_mask = labels.eq(-100) # B x T
# Makes the -100 to 0
labels = torch.clamp(labels, min=0)
# Gather ???? at dims when labels are non zero?
nll_loss = log_probs.gather(dim=-1, index=labels) # Gather the log probs at the label indices over the vocabulary dimension
nll_loss.masked_fill_(padding_mask, 0.0)
# Modification to add RWR term??? padding_mask - can this be point multiplied with the rewards?
# reward_padding_mask = padding_mask * torch.exp(rewards/0.9)
# Multiply by the RWR term
print(nll_loss)
nll_loss = nll_loss * torch.exp(rewards/0.9)
# Take the mean over the label dimensions, then divide by the number of active elements (i.e. not-padded):
num_active_elements = padding_mask.numel() - padding_mask.long().sum()
nll_loss = nll_loss.sum() / num_active_elements

print(nll_loss)

tensor([[[21.0105],
         [21.0105],
         [21.0105],
         ...,
         [21.0105],
         [21.0105],
         [21.0105]],

        [[21.0105],
         [21.0105],
         [21.0105],
         ...,
         [21.0105],
         [21.0105],
         [21.0105]]], grad_fn=<MaskedFillBackward0>)
tensor(63.8243, grad_fn=<DivBackward0>)


In [60]:
63.8243/3.0377

21.010731803667248

In [58]:
num_active_elements

tensor(8190)

In [49]:
padding_mask

tensor([[[False],
         [False],
         [False],
         ...,
         [False],
         [False],
         [False]],

        [[False],
         [False],
         [False],
         ...,
         [False],
         [False],
         [False]]])

In [50]:
torch.exp(rewards/0.9)

tensor([[[1.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]],

        [[1.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]]])

In [48]:
reward_padding_mask

tensor([[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]])

In [45]:
nll_loss

tensor(21.0105, grad_fn=<DivBackward0>)

In [19]:
print(labels.shape)

torch.Size([2, 4095])


In [17]:
import torch.nn.functional as F

log_probs = -F.log_softmax(logits, dim=-1)

In [18]:
log_probs.shape

torch.Size([2, 4095, 32000])

In [20]:
if labels.dim() == log_probs.dim() - 1:
    labels = labels.unsqueeze(-1)

In [21]:
labels.shape

torch.Size([2, 4095, 1])

In [22]:
nll_loss = log_probs.gather(dim=-1, index=labels)

In [23]:
nll_loss.shape

torch.Size([2, 4095, 1])

In [24]:
nll_loss.mean()

tensor(21.0105, grad_fn=<MeanBackward0>)

In [30]:
from torch.nn import CrossEntropyLoss

In [33]:
logits.shape

torch.Size([2, 4095, 32000])

ValueError: Expected input batch_size (8190) to match target batch_size (8192).

In [29]:
labels.eq(1.).shape

torch.Size([2, 4095, 1])

In [26]:
log_probs.gather(dim=-1, index =torch.ones(size = (2, 4096)).long().view(-1, 1)).shape

RuntimeError: Index tensor must have the same number of dimensions as input tensor

In [82]:
model.parameters

<bound method Module.parameters of MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): Mistr

In [9]:
type(out)

transformers.modeling_outputs.CausalLMOutputWithPast

In [17]:
out["logits"].shape
# B x input_len x embed_dim

torch.Size([2, 4096, 32000])

In [20]:
import torch.nn as nn
log_probs = -nn.functional.log_softmax(out["logits"], dim=-1)

In [26]:
labels = torch.ones(size = (2, 4096,32000)).long()

In [27]:


# labels = torch.clamp(labels, min=0)
nll_loss = log_probs.gather(dim=-1, index=labels)

In [28]:
log_probs.shape # B x fixed_sequence_length x vocabulary_size

torch.Size([2, 4096, 32000])

In [29]:
labels.shape

torch.Size([2, 4096, 32000])

In [30]:
nll_loss.shape  # B x fixed_sequence_length x vocabulary_size


torch.Size([2, 4096, 32000])

In [31]:
log_probs.sum(dim=-1, keepdim=True, dtype=torch.float32).shape

torch.Size([2, 4096, 1])

In [32]:
nll_loss.shape

torch.Size([2, 4096, 32000])

In [41]:
pm = torch.tensor([1., 0.]).view(-1, 1, 1)

nll_loss *pm 

tensor([[[21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         ...,
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105],
         [21.0105, 21.0105, 21.0105,  ..., 21.0105, 21.0105, 21.0105]],

        [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<MulBackward0>)

In [35]:
nll_loss.shape

torch.Size([2, 4096, 32000])

In [36]:
torch.tensor([1., 0.]).shape

torch.Size([2])

In [37]:
pm = torch.tensor([1., 0.]).view(-1, 1, 1)


In [39]:
pm.shape

torch.Size([2, 1, 1])

In [73]:
from fastchat.model.model_adapter import get_conversation_template

def non_system_messages(raw_msg):
    conv = get_conversation_template("gpt-4")
    for msg in raw_msg:
        if msg["role"] == "system":
            continue
        conv.append_message(role=msg["role"], message=msg["content"])
    return conv.to_openai_api_messages()[1:]  # remove system prompt

In [83]:
c= torch.ones((3, 3))

In [85]:
c[0] = 0

In [86]:
c

tensor([[0., 0., 0.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [87]:
torch.exp(c)

tensor([[1.0000, 1.0000, 1.0000],
        [2.7183, 2.7183, 2.7183],
        [2.7183, 2.7183, 2.7183]])